# 00 — Audit dataset dan 5-fold split

CPU runtime cukup. Membaca ZIP dari Drive, mengekstrak secara aman ke SSD Colab, memeriksa citra, dan menghasilkan manifest/splits bebas benturan SHA256.

**Pipeline revisi untuk artikel ilmiah.** Hasil tidak otomatis identik dengan skripsi 2026 karena split, kontrol, dan beberapa transformasi diperbarui. Jalankan notebook menurut urutan nomor.

In [ ]:
from google.colab import drive
from pathlib import Path
import sys, os

drive.mount('/content/drive')
PROJECT=Path(os.environ.get('FFE_PROJECT','/content/drive/MyDrive/tugas_akhir/FFE_Publication_2026'))
assert PROJECT.exists(), f'Folder proyek tidak ditemukan: {PROJECT}'
os.environ['FFE_ROOT']=str(PROJECT)
sys.path.insert(0,str(PROJECT/'02_notebooks'))
from ffe_core import ROOT, paths, seed_all, ensure_dataset_local, load_manifest, load_split
P=paths(); seed_all()
print('Project:',ROOT)

## 1. Ekstraksi dari ZIP sumber (tidak memodifikasi arsip)
Perlu penyimpanan lokal Colab yang cukup. File hasil ekstraksi tidak dijadikan sumber permanen; manifest menyimpan path relatif agar notebook bisa dijalankan ulang.

In [ ]:
import pandas as pd, numpy as np, json, hashlib
from ffe_core import ZIP, LOCAL, image_records, prepare_splits, test_group_integrity, CLASS_NAMES
assert ZIP.exists(), f'ZIP not found: {ZIP}'
print('ZIP:',ZIP,'bytes:',ZIP.stat().st_size)
print('SHA256 archive:',hashlib.sha256(ZIP.read_bytes()).hexdigest())
ensure_dataset_local()
df=image_records()
print('Jumlah citra valid:',len(df),'jumlah spesies:',df.species.nunique())
display(pd.crosstab(df.species,df.freshness).reindex(columns=CLASS_NAMES))
df.to_csv(P['manifests']/'dataset_manifest.csv',index=False)

## 2. Audit identitas gambar
SHA256 mendeteksi gambar identik byte-per-byte. pHash hanya indikator visual kasar, **bukan** ID ikan atau dasar membuang otomatis gambar. Jika ada citra dari ikan yang sama pada beberapa hari, group ID harus diverifikasi dari metadata sumber.

In [ ]:
exact=df.groupby('sha256').agg(n=('sample_id','size'),labels=('freshness','nunique'),species=('species','nunique')).query('n>1')
exact.to_csv(P['manifests']/'exact_duplicates.csv')
print('Grup duplikasi persis:',len(exact),'baris terlibat:',int(exact.n.sum()) if len(exact) else 0)
ph=df.groupby('phash').agg(n=('sample_id','size'),examples=('rel_path','first')).query('n>1')
ph.to_csv(P['manifests']/'phash_matches.csv')
print('Grup pHash identik:',len(ph),'(perlu pemeriksaan manual)')
counts=df.groupby(['species','freshness']).size().rename('n').reset_index()
counts.to_csv(P['manifests']/'class_counts.csv',index=False)
print('Dataset original menurut dokumentasi repositori Mendeley: 4.392; bila jumlah di sini berbeda, laporkan selisih, jangan diam-diam hapus data.')
if len(df)!=4392:print('PERINGATAN: jumlah valid sekarang',len(df),'— verifikasi dengan daftar asli dan jurnal data.')

## 3. Outer 5 folds + inner validation
Gunakan stratifikasi gabungan species × freshness; kelompok dengan SHA256 sama tidak boleh terpisah. Ini **belum** membuktikan independensi ikan individual. Jika tersedia fish_id/session_id, ganti kolom grouping dan ulang split.

In [ ]:
prepare_splits(df,n_splits=5,seed=42)
assert test_group_integrity()
summary=[]
for fold in range(1,6):
 for part in ('train','val','test'):
  d=load_split(fold,part)
  summary.append({'fold':fold,'partition':part,'n':len(d), **{f'n_{k}':int((d.freshness==k).sum()) for k in CLASS_NAMES}})
display(pd.DataFrame(summary))
pd.DataFrame(summary).to_csv(P['manifests']/'split_counts.csv',index=False)

## 4. Reproducibility manifest dan protokol
Data test **tidak** digunakan untuk hyperparameter, seleksi fitur, scaler, ataupun early stopping.

In [ ]:
import platform
import sklearn, cv2
protocol={'dataset_zip':ZIP.name,'zip_sha256':hashlib.sha256(ZIP.read_bytes()).hexdigest(),
 'n_images':len(df),'seed':42,'outer_cv':'StratifiedGroupKFold(5), 24 strata, SHA256 groups',
 'inner_cv':'StratifiedGroupKFold(5) first split on outer train portion',
 'class_names':CLASS_NAMES,'python':platform.python_version(),'sklearn':sklearn.__version__,'opencv':cv2.__version__,
 'warning':'Image-level SHA256 groups do not establish fish/session independence.'}
(P['manifests']/'protocol.json').write_text(json.dumps(protocol,indent=2))
print('Notebook 00 selesai; manifest, audit, dan 15 split CSV tersimpan.')